# CIFAR-100 Multi-Seed Runner — Dissociation Trio (no-MoE)

Dispatches the 9 CIFAR-100 runs that back the paper's dissociation
claim on CIFAR-100: **3 variants × 3 seeds**, all at the no-MoE config
recommended by the CIFAR-10 component ablation.

| Group | Variants | Seeds |
|---|---|---|
| Cumulative baseline | `γ = 0.7` | 42, 123, 456 |
| γ = 0 (purely local) | `γ = 0.0` | 42, 123, 456 |
| Hardness-gated κ = 0 | adaptive, `γ₀ = 0.7`, `τ = 1.0` | 42, 123, 456 |

All runs use the trainer
`cp_fair_cifar100.py`, which reads the
full env-var interface: `FF_GAMMA_SCALE`, `FF_GAMMA_GATING_MODE`,
`FF_GAMMA_0`, `FF_GAMMA_KAPPA`, `FF_GAMMA_TAU`, **`FF_N_EXPERTS=1`**,
**`FF_MOE_TOP_K=1`**, `FF_SEED`.

The existing 3 CIFAR-100 baseline seeds on Drive were trained with
MoE on (`n_experts=32, top_k=4`). Those remain a **secondary**
with-MoE comparison in the paper; this notebook produces the primary
matched dissociation trio.

Checkpoints and metrics are written to Drive every 60 s via a
background syncer so a Colab disconnect never loses progress.
Re-running the notebook resumes from whatever is on Drive.

### Prerequisite — upload the trainer to Drive

Before running this notebook, upload the trainer (with the new
`FF_N_EXPERTS` / `FF_MOE_TOP_K` bindings) to:

```
/content/drive/MyDrive/ff_cifar100_multiseed/cp_fair_cifar100.py
```

Source:
`supplement_code/trainers/cp_fair_cifar100.py`

If you uploaded an earlier copy before the MoE patch, **replace it** —
the older trainer ignores `FF_N_EXPERTS` and will fall back to default settings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, subprocess, shutil, time, threading, sys
from datetime import datetime

DRIVE_BASE = '/content/drive/MyDrive/ff_cifar100_multiseed'
WORK_DIR = '/content/ff_cifar100'
SCRIPT_NAME = 'cp_fair_cifar100.py'
SCRIPT_DRIVE = f'{DRIVE_BASE}/{SCRIPT_NAME}'
SCRIPT_LOCAL = f'{WORK_DIR}/{SCRIPT_NAME}'
PROGRESS_FILE = f'{DRIVE_BASE}/training_progress.json'

os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.exists(SCRIPT_DRIVE), (
    f'Upload the trainer to {SCRIPT_DRIVE} before running.'
)
shutil.copy2(SCRIPT_DRIVE, SCRIPT_LOCAL)
print(f'Drive base:  {DRIVE_BASE}')
print(f'Work dir:    {WORK_DIR}')
print(f'Script:      {SCRIPT_LOCAL}')


## 1. Run Configs (A4 + A5)

Each row is one run. Shared env vars (architecture, epochs) go in
`COMMON_ENV`. Per-run overrides go in each config's `env` dict.

The CIFAR-100 baseline at γ=0.7 already has 3 seeds (123, 456, 789);
those runs are therefore **not** in this list.

In [ ]:
# ===================== CHANGE HERE IF NEEDED =====================
# No-MoE (dense MLP) config — matches the paper's component-ablation
# recommendation (MoE is +0.17pp when removed) and the Tiny-ImageNet
# recipe. The existing CIFAR-100 baseline seeds on Drive were trained
# with n_experts=32, top_k=4; those remain a secondary with-MoE
# comparison. This new trio is the primary dissociation evidence.
#
# NOTE: FF_N_EXPERTS and FF_MOE_TOP_K require the trainer patched to
# read them (added in supplement_code/trainers/ — re-upload the patched .py
# to Drive before running).
COMMON_ENV = {
    'FF_NUM_BLOCKS':    '4',
    'FF_D_MODEL':       '256',
    'FF_NUM_HEADS':     '8',
    'FF_EPOCHS_STAGE1': '362',
    'FF_USE_SAM':       '0',
    # No MoE: dense MLP as recommended by the CIFAR-10 component ablation
    'FF_N_EXPERTS':     '1',
    'FF_MOE_TOP_K':     '1',
}

# The matched dissociation trio at no-MoE config:
#   - cumulative gamma=0.7 (new matched baseline; rerun because existing
#     baseline seeds on Drive use MoE=on)
#   - gamma=0 (purely local)
#   - hardness-gated kappa=0 (gamma_0=0.7, tau=1.0, adaptive)
# 3 seeds per variant = 9 runs total. At ~2 min/epoch, each run ~12 h.
RUN_CONFIGS = [
    # --- Cumulative baseline (no-MoE) ---
    ('cumulative_seed42',   {'FF_GAMMA_SCALE': '0.7', 'FF_SEED': '42'}),
    ('cumulative_seed123',  {'FF_GAMMA_SCALE': '0.7', 'FF_SEED': '123'}),
    ('cumulative_seed456',  {'FF_GAMMA_SCALE': '0.7', 'FF_SEED': '456'}),
    # --- gamma=0 (purely local) ---
    ('gamma0_seed42',   {'FF_GAMMA_SCALE': '0.0', 'FF_SEED': '42'}),
    ('gamma0_seed123',  {'FF_GAMMA_SCALE': '0.0', 'FF_SEED': '123'}),
    ('gamma0_seed456',  {'FF_GAMMA_SCALE': '0.0', 'FF_SEED': '456'}),
    # --- Hardness-gated kappa=0 ---
    ('gated_k0_seed42',  {
        'FF_GAMMA_GATING_MODE': 'adaptive',
        'FF_GAMMA_0':    '0.7',
        'FF_GAMMA_KAPPA':'0.0',
        'FF_GAMMA_TAU':  '1.0',
        'FF_SEED':       '42',
    }),
    ('gated_k0_seed123', {
        'FF_GAMMA_GATING_MODE': 'adaptive',
        'FF_GAMMA_0':    '0.7',
        'FF_GAMMA_KAPPA':'0.0',
        'FF_GAMMA_TAU':  '1.0',
        'FF_SEED':       '123',
    }),
    ('gated_k0_seed456', {
        'FF_GAMMA_GATING_MODE': 'adaptive',
        'FF_GAMMA_0':    '0.7',
        'FF_GAMMA_KAPPA':'0.0',
        'FF_GAMMA_TAU':  '1.0',
        'FF_SEED':       '456',
    }),
]

print(f'{len(RUN_CONFIGS)} runs queued (no-MoE config):')
for tag, env in RUN_CONFIGS:
    if 'FF_GAMMA_GATING_MODE' in env:
        mode = f"gated kappa={env['FF_GAMMA_KAPPA']}"
    elif env.get('FF_GAMMA_SCALE') == '0.0':
        mode = 'gamma=0'
    else:
        mode = f"cumulative gamma={env['FF_GAMMA_SCALE']}"
    print(f'  - {tag:24s}  seed={env["FF_SEED"]:<4}  {mode}')


## 2. Helpers (progress, accuracy parsing, done-detection)

In [ ]:
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {}

def save_progress(p):
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(p, f, indent=2)

def extract_accuracy(log_path):
    """Parse final test accuracy (Top1, Top5) for Stage-1/Stage-2 from the log."""
    r = {}
    if not os.path.exists(log_path):
        return r
    with open(log_path) as f:
        lines = f.readlines()
    for line in reversed(lines):
        if 'Stage-2 Head' in line and 'Top1=' in line:
            try:
                top1 = line.split('Top1=')[1].split('%')[0] + '%'
                top5 = line.split('Top5=')[1].split('%')[0] + '%'
                if 'no-TTA' in line:
                    r['s2_top1_notta'] = top1; r['s2_top5_notta'] = top5
                elif 'TTA' in line:
                    r['s2_top1_tta'] = top1; r['s2_top5_tta'] = top5
            except Exception:
                pass
        if 'Stage-1 FF' in line and 'TTA' in line and 'no-TTA' not in line and 'Top1=' in line:
            try:
                r['s1_top1_tta'] = line.split('Top1=')[1].split('%')[0] + '%'
                r['s1_top5_tta'] = line.split('Top5=')[1].split('%')[0] + '%'
            except Exception:
                pass
        if len(r) >= 6:
            break
    return r

def is_done(tag):
    ckpt_dir = f'{DRIVE_BASE}/checkpoints_{tag}'
    if os.path.exists(ckpt_dir):
        if any('stage2_best' in f for f in os.listdir(ckpt_dir)):
            return True
    log = f'{DRIVE_BASE}/{tag}_stdout.log'
    return 's2_top1_tta' in extract_accuracy(log) if os.path.exists(log) else False

# Status of each run
progress = load_progress()
print('Status of all 6 runs:')
for tag, _ in RUN_CONFIGS:
    status = 'DONE' if is_done(tag) else progress.get(tag, {}).get('status', 'pending')
    acc = progress.get(tag, {}).get('s2_top1_tta', '---')
    print(f'  {tag:22s} [{status:<10s}] S2-Top1-TTA: {acc}')


## 3. DriveSyncer + Training Loop

`DriveSyncer` is a background thread that copies local JSONL/JSON/NPZ
metrics to Drive every 60s. Checkpoints are written directly to Drive
by the trainer (via `FF_CKPT_DIR`). This survives a Colab disconnect:
on re-run, `is_done(tag)` sees the `stage2_best.pt` on Drive and skips
the run.

In [ ]:
class DriveSyncer(threading.Thread):
    def __init__(self, local_dir, drive_dir, log_local, log_drive, interval=60):
        super().__init__(daemon=True)
        self.local_dir = local_dir
        self.drive_dir = drive_dir
        self.log_local = log_local
        self.log_drive = log_drive
        self.interval = interval
        self._stop = threading.Event()

    def _sync(self):
        try:
            for root, dirs, files in os.walk(self.local_dir):
                rel = os.path.relpath(root, self.local_dir)
                dst_root = os.path.join(self.drive_dir, rel)
                os.makedirs(dst_root, exist_ok=True)
                for fn in files:
                    if fn.endswith(('.jsonl', '.json', '.npz')):
                        shutil.copy2(os.path.join(root, fn), os.path.join(dst_root, fn))
            if os.path.exists(self.log_local):
                shutil.copy2(self.log_local, self.log_drive)
        except Exception as e:
            print(f'[DriveSyncer] sync error (will retry): {e}')

    def run(self):
        while not self._stop.is_set():
            self._stop.wait(self.interval)
            if not self._stop.is_set():
                self._sync()

    def stop(self):
        self._stop.set()
        self._sync()


def run_one(tag, env_overrides):
    ckpt_dir  = f'{DRIVE_BASE}/checkpoints_{tag}'
    runs_dir  = f'{WORK_DIR}/runs_{tag}'                # LOCAL (fast writes)
    runs_drv  = f'{DRIVE_BASE}/runs_{tag}'              # DRIVE (sync target)
    log_drv   = f'{DRIVE_BASE}/{tag}_stdout.log'
    log_loc   = f'{WORK_DIR}/{tag}_stdout.log'
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(runs_dir, exist_ok=True)
    os.makedirs(runs_drv, exist_ok=True)

    env = os.environ.copy()
    env.update(COMMON_ENV)
    env.update(env_overrides)
    env['FF_VERSION_TAG'] = f'cifar100_{tag}'
    env['FF_RUNS_DIR']    = runs_dir
    env['FF_CKPT_DIR']    = ckpt_dir

    existing = [f for f in os.listdir(ckpt_dir) if f.endswith('.pt')]
    print(f'\n{"=" * 64}')
    print(f'  CIFAR-100 Multi-Seed — {tag}')
    seed = env_overrides.get('FF_SEED')
    mode = 'γ=0' if env_overrides.get('FF_GAMMA_SCALE') == '0.0' else f"gated κ={env_overrides.get('FF_GAMMA_KAPPA')}"
    print(f'  Mode:         {mode}, seed={seed}')
    print(f'  Checkpoints:  {ckpt_dir} (Drive)')
    print(f'  Metrics:      {runs_dir} (local) → {runs_drv} every 60s')
    print(f'  Existing ckpts: {existing or "(none, starting fresh)"}')
    print(f'{"=" * 64}\n')

    p = load_progress()
    p[tag] = {
        'status': 'running',
        'started': datetime.now().isoformat(),
        'seed': seed,
        'env': {k: v for k, v in env_overrides.items()},
    }
    save_progress(p)

    syncer = DriveSyncer(runs_dir, runs_drv, log_loc, log_drv, interval=60)
    syncer.start()

    start_time = time.time()
    with open(log_loc, 'a') as lf:
        proc = subprocess.Popen(
            [sys.executable, '-u', SCRIPT_LOCAL], env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
            cwd=WORK_DIR,
        )
        for line in proc.stdout:
            print(line, end='')
            lf.write(line); lf.flush()
        proc.wait()
    elapsed = time.time() - start_time
    syncer.stop()

    acc = extract_accuracy(log_loc)
    p = load_progress()
    p[tag] = {
        'status': 'done' if acc else 'incomplete',
        'seed': seed,
        'finished': datetime.now().isoformat(),
        'elapsed_sec': round(elapsed, 1),
        'return_code': proc.returncode,
        **acc,
    }
    save_progress(p)
    print(f'\n>>> {tag}: {acc or f"rc={proc.returncode}"} (elapsed: {elapsed / 60:.1f} min)')
    return acc


## 4. Run all 6 configs sequentially (skipping done ones)

Re-executing this cell after a disconnect resumes from Drive state: any
run whose `stage2_best.pt` exists on Drive is skipped.

In [ ]:
for tag, env_overrides in RUN_CONFIGS:
    if is_done(tag):
        acc = extract_accuracy(f'{DRIVE_BASE}/{tag}_stdout.log')
        print(f'[skip] {tag}: already complete — {acc}')
        continue
    p = load_progress()
    if p.get(tag, {}).get('status') == 'running':
        print(f'[resume] {tag}: previous run crashed or disconnected — restarting from last checkpoint.')
    run_one(tag, env_overrides)


## 5. Summary table

In [ ]:
import numpy as np

def _num(s, default=None):
    try:
        return float(s.rstrip('%')) if isinstance(s, str) else default
    except Exception:
        return default

progress = load_progress()
rows = []
for tag, _ in RUN_CONFIGS:
    r = progress.get(tag, {})
    rows.append((tag,
                 r.get('status', 'pending'),
                 r.get('s1_top1_tta', '---'),
                 r.get('s2_top1_notta', '---'),
                 r.get('s2_top1_tta', '---')))

print(f'{"run":24s} {"status":<10s} {"S1-TTA":>8s} {"S2-noTTA":>10s} {"S2-TTA":>8s}')
print('-' * 68)
for r in rows:
    print(f'{r[0]:24s} {r[1]:<10s} {r[2]:>8s} {r[3]:>10s} {r[4]:>8s}')

# Compute mean+-std for each group (cumulative / gamma0 / gated)
print('\n--- group summaries (S2 TTA) ---')
for prefix in ('cumulative_seed', 'gamma0_seed', 'gated_k0_seed'):
    vals = []
    for tag, _ in RUN_CONFIGS:
        if tag.startswith(prefix):
            v = _num(progress.get(tag, {}).get('s2_top1_tta'))
            if v is not None:
                vals.append(v)
    if vals:
        mu = float(np.mean(vals))
        sd = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        print(f'  {prefix[:-5]:<12s}  n={len(vals)}  mean={mu:.2f}%  std={sd:.2f}%')
    else:
        print(f'  {prefix[:-5]:<12s}  (no results yet)')
